In [2]:
import pandas as pd
from rouge_score import rouge_scorer
from summac.model_summac import SummaCZS
import json
import re
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')


/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/python/3.12.1/lib/python3.12/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
[nltk_data] Downloading package punkt to /home/codespace/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
with open("movie_info.json", "r") as f:
    data = json.load(f)

rows = []

for key, value in data.items():
    path = value["subtitle_path"]
    folder = path.split("/")[-2]
    movie = re.sub(r"\(\d{4}\)", "", folder)
    movie = movie.replace(".", " ").strip()
    movie = movie.title()
    rows.append({
        "movie_id": key,
        "movieName": movie
    })

# Create DataFrame
movies = pd.DataFrame(rows)
movies['movie_id'] = movies['movie_id'].astype("Int64")
ds_df = pd.read_csv("deepSeekFINAL.csv")
ll_df = pd.read_csv("llamaPromptedSummaries.csv")
ll_df = ll_df.rename(columns={'prompted_summary' : 'generated_summary'})
ll_df.head()

,movie_id,subtitles,summaries,generated_summary
0,9249578,~A TOKUMA SHOTEN PRODUCTION~ Ha ha ha! - Ha h...,"In the film's backstory, human civilizations b...","The movie appears to be ""Castle in the Sky"" (,..."
1,9226114,I learned a word after I got here. This word h...,"Miu, a fishmonger at the Prosperity Market. Be...",The movie appears to be a comedy-drama that fo...
2,9395989,(Helicopter whirring) Narrator: Virtually. the...,The film opens on Stewart Graff jogging undern...,"The movie ""Earthquake"" is a disaster film that..."
3,9447919,[SINGING] ♪ It's all right ♪ For tonight. To b...,Stewart 'P.C.' Simpson lives in a magnificent ...,The plot of the movie revolves around the life...
4,9224567,. That's right. That's right. Come on. Come o...,Michael Dorsey is a respected but perfectionis...,"The movie follows the story of Michael Dorsey,..."


In [4]:
## Compute Rouge for both Models
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def get_rouge(row):
    score = scorer.score(str(row['summaries']), str(row['generated_summary']))
    
    return pd.Series({
        'rouge1': score['rouge1'].fmeasure,
        'rouge2': score['rouge2'].fmeasure,
        'rougeL': score['rougeL'].fmeasure
    })

ds_df[['rouge1', 'rouge2', 'rougeL']] = ds_df.apply(get_rouge, axis=1)
ll_df[['rouge1', 'rouge2', 'rougeL']] = ll_df.apply(get_rouge, axis=1)


In [ ]:
model = SummaCZS(granularity="sentence", model_name="vitc", device="cpu")

ds_df["summac"] = model.score(
    ds_df["subtitles"].astype(str).tolist(),
    ds_df["generated_summary"].astype(str).tolist()
)["scores"]

ll_df["summac"] = model.score(
    ll_df["subtitles"].astype(str).tolist(),
    ll_df["generated_summary"].astype(str).tolist()
)["scores"]

/usr/local/python/3.12.1/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [ ]:
ds_results = ds_df[['movie_id', 'rouge1', 'rouge2', 'rougeL', 'summac']]
ds_results = ds_results.merge(movies, on ='movie_id', how='left')
ds_results.sort_values('rouge1', ascending=False).head(5)


,movie_id,rouge1,rouge2,rougeL,movieName
149,9506982,0.533333,0.189415,0.252778,Cannibal Women In The Avocado Jungle Of Death
231,9376852,0.527660,0.149573,0.229787,The Wildest Dream
83,9312229,0.496711,0.118812,0.230263,The Flesh Eaters
140,9310280,0.485535,0.090794,0.186164,The Phantom Planet
314,9502507,0.479433,0.071124,0.195745,Dangerous Beauty


In [ ]:
ds_results.sort_values('rouge2', ascending=False).head(5)

,movie_id,rouge1,rouge2,rougeL,movieName
149,9506982,0.533333,0.189415,0.252778,Cannibal Women In The Avocado Jungle Of Death
231,9376852,0.527660,0.149573,0.229787,The Wildest Dream
157,9212873,0.421365,0.137313,0.225519,The Crossing
73,9403636,0.452703,0.125424,0.195946,Bambi
347,9489880,0.366581,0.124680,0.168798,Shrek


In [ ]:
ds_results.sort_values('rougeL', ascending=False).head(5)

,movie_id,rouge1,rouge2,rougeL,movieName
149,9506982,0.533333,0.189415,0.252778,Cannibal Women In The Avocado Jungle Of Death
210,9263407,0.421053,0.095238,0.236842,Recount
36,9221113,0.455696,0.119782,0.235081,Salt For Svanetia
263,9264457,0.387597,0.085938,0.232558,The Naked Jungle
296,9200491,0.371901,0.091667,0.231405,The Mad Doctor


In [ ]:
ll_results = ll_df[['movie_id', 'rouge1', 'rouge2', 'rougeL', 'summac']]
ll_results = ll_results.merge(movies, on ='movie_id', how='left')
ll_results.sort_values('rouge1', ascending=False).head(5)

,movie_id,rouge1,rouge2,rougeL,movieName
267,9503643,0.550523,0.155995,0.204413,Imitation Of Life
149,9506982,0.534819,0.175978,0.236769,Cannibal Women In The Avocado Jungle Of Death
36,9221113,0.533808,0.175000,0.263345,Salt For Svanetia
115,9286226,0.532775,0.145455,0.239888,Silverado
73,9403636,0.532526,0.133536,0.226929,Bambi


In [ ]:
ll_results.sort_values('rouge2', ascending=False).head(5)

,movie_id,rouge1,rouge2,rougeL,movieName
149,9506982,0.534819,0.175978,0.236769,Cannibal Women In The Avocado Jungle Of Death
36,9221113,0.533808,0.175000,0.263345,Salt For Svanetia
77,9242689,0.469799,0.168161,0.230425,The Return Of The Pink Panther
122,9512869,0.429947,0.160772,0.233155,Cinderella
327,9489767,0.467440,0.159071,0.206958,Star Trek Nemesis


In [ ]:
ll_results.sort_values('rougeL', ascending=False).head(5)

,movie_id,rouge1,rouge2,rougeL,movieName
36,9221113,0.533808,0.175000,0.263345,Salt For Svanetia
8,9507853,0.380488,0.117647,0.253659,Carry On Spying
100,9470963,0.449649,0.131765,0.252927,The Bachelor Father
115,9286226,0.532775,0.145455,0.239888,Silverado
149,9506982,0.534819,0.175978,0.236769,Cannibal Women In The Avocado Jungle Of Death


In [ ]:
ds_results[['rouge1', 'rouge2', 'rougeL', 'summac']].describe(include='all')

,rouge1,rouge2,rougeL
count,397.000000,397.000000,397.000000
mean,0.302346,0.060843,0.152018
std,0.087682,0.029304,0.034590
min,0.043551,0.000000,0.033501
25%,0.241772,0.038554,0.128788
50%,0.303371,0.058568,0.153191
75%,0.373832,0.079351,0.174089
max,0.533333,0.189415,0.252778


In [ ]:
ll_results[['rouge1', 'rouge2', 'rougeL', 'summac']].describe(include='all')

KeyError: "['summac'] not in index"